### Now that we know, dates play a large role in the dropping of our scores lets try this again without any dates

# import libraries

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time, math
import os
import re

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer # applies dif preprocessing operations to dif columns in dataset
from sklearn.impute import SimpleImputer # fills in missing values in dataset using strats like mean, median, or most frequent
from sklearn.pipeline import Pipeline # chains multiple preprocessing steps and models together into a single object
from sklearn.preprocessing import OneHotEncoder, StandardScaler # converts categorical variables to seperate binary / scaler: standardizes numerical features so they have a mean of 0 and std of 1
from sklearn.linear_model import LogisticRegression # lets u create a logistic regression model for binary classification tasks
from sklearn.dummy import DummyClassifier # create a baseline model that makes predictions based on simple rules or random chance
from sklearn.metrics import accuracy_score, precision_score, recall_score, average_precision_score, f1_score, confusion_matrix
# just dif ways to measure how well a classification model is performing
from sklearn.base import clone

# Load and Clean 

In [4]:
df = pd.read_csv('../data/fraud_oracle.csv') # since they are in the both folder, we need to go one level up to access the data folder and then access the csv file

df = df[df['MonthClaimed'] != '0'].copy()
assert len(df) == 15419

# dealing with Age
df['AgeMissing'] = (df['Age'] == 0).astype(int) # create a new column called AgeMissing that is 1 if Age is 0 and 0 otherwise
df['Age'] = df['Age'].replace(0, np.nan) # replace 0s in Age with NaN values

In [5]:
# check cell
assert len(df) == 15419, f"Expected 15,419 rows after dropping the corrupt row, got {len(df)}"
assert (df['MonthClaimed'] != '0').all() and (df['DayOfWeekClaimed'] != '0').all(), "A '0' claim date is still present"
assert df['AgeMissing'].sum() == 319, f"Expected 319 unknown ages, got {df['AgeMissing'].sum()}"
assert df.loc[df['AgeMissing'] == 1, 'Age'].isna().all(), "Unknown ages weren't set to NaN"
assert df['Age'].dropna().min() > 0, "An Age of 0 is still in the data"
print(f"✓ Step 1 | fraud cases: {df['FraudFound_P'].sum()} ({df['FraudFound_P'].mean():.2%})")

✓ Step 1 | fraud cases: 923 (5.99%)


# step 2: new and improved column list --> no dates

In [6]:
numeric_cols = ['Age', 'Deductible', 'VehiclePrice', 'Days_Policy_Accident',
                'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle',
                'AgeOfPolicyHolder', 'NumberOfCars', 'AgeMissing']
categorical_cols = ['Make', 'AccidentArea', 'Sex', 'MaritalStatus', 'Fault',
                    'VehicleCategory', 'DriverRating', 'AgentType', 'BasePolicy']
features = numeric_cols + categorical_cols

# Dropped for leakage or redundancy (see data dictionary)
leakage_dropped = ['PolicyNumber', 'RepNumber', 'PolicyType', 'PoliceReportFiled',
                   'WitnessPresent', 'NumberOfSuppliments', 'AddressChange_Claim']
# Dropped for hurting validation: safe at filing, but caught 37.5% @ top 20%
# with them vs 53.5% without (see experiment notebook, Check 4)
date_dropped = ['Month', 'WeekOfMonth', 'DayOfWeek',
                'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed']

In [7]:
# step 2 check 
dropped = set(leakage_dropped) | set(date_dropped)
assert not set(numeric_cols) & set(categorical_cols), "A column is listed as both numeric and categorical"
assert not set(features) & dropped, f"Dropped column still in features: {set(features) & dropped}"
assert 'FraudFound_P' not in features and 'Year' not in features, "Label or Year is in the features"
assert set(features) <= set(df.columns), f"Missing columns: {set(features) - set(df.columns)}"
assert len(features) == len(set(features)), "Duplicate column in features"
assert 'Fault' in features and 'AgeOfPolicyHolder' in features, "Fault / AgeOfPolicyHolder should be kept"
print(f"✓ Step 2 | {len(numeric_cols)} numeric + {len(categorical_cols)} categorical = {len(features)} features")

✓ Step 2 | 10 numeric + 9 categorical = 19 features


# step 3: turn text ranges into numbers

In [8]:
import re

def parse_numeric_category(value, column):
    if pd.isna(value):
        return np.nan, "missing"

    s = str(value).strip().lower()

    # Special words with no digits
    if s == "none":
        return 0.0, "none"
    if s == "new":
        return 0.0, "new"

    # Ranges: "26 to 30", "3 to 4" → midpoint
    m = re.fullmatch(r"(\d+(?:\.\d+)?)\s*to\s*(\d+(?:\.\d+)?)(?:\s+\w+)?", s)
    if m:
        low, high = map(float, m.groups())
        return (low + high) / 2, "range"

    # Open-ended: "more than 7" → 8, "over 65" → 66, "less than 20000" → 19999
    m = re.fullmatch(r"(more than|less than|over)\s+(\d+(?:\.\d+)?)(?:\s+\w+)?", s)
    if m:
        direction, number = m.groups()
        number = float(number)
        if direction in ("more than", "over"):
            return number + 1, direction.replace(" ", "_")
        return number - 1, direction.replace(" ", "_")

    # Exact: "1", "2 years", "1 vehicle"
    m = re.fullmatch(r"(\d+(?:\.\d+)?)(?:\s+\w+)?", s)
    if m:
        return float(m.group(1)), "exact"

    raise ValueError(f"Unrecognized value {value!r} in column {column!r}")

In [9]:
# preview mapping (run before converting all columns to numeric)
range_cols = ['VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim',
              'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'NumberOfCars']

for col in range_cols:
    mapping = {v: parse_numeric_category(v, col)[0] for v in sorted(df[col].unique())}
    print(col, mapping)

n_categories_before = df[range_cols].nunique()

VehiclePrice {'20000 to 29000': 24500.0, '30000 to 39000': 34500.0, '40000 to 59000': 49500.0, '60000 to 69000': 64500.0, 'less than 20000': 19999.0, 'more than 69000': 69001.0}
Days_Policy_Accident {'1 to 7': 4.0, '15 to 30': 22.5, '8 to 15': 11.5, 'more than 30': 31.0, 'none': 0.0}
Days_Policy_Claim {'15 to 30': 22.5, '8 to 15': 11.5, 'more than 30': 31.0}
PastNumberOfClaims {'1': 1.0, '2 to 4': 3.0, 'more than 4': 5.0, 'none': 0.0}
AgeOfVehicle {'2 years': 2.0, '3 years': 3.0, '4 years': 4.0, '5 years': 5.0, '6 years': 6.0, '7 years': 7.0, 'more than 7': 8.0, 'new': 0.0}
AgeOfPolicyHolder {'16 to 17': 16.5, '18 to 20': 19.0, '21 to 25': 23.0, '26 to 30': 28.0, '31 to 35': 33.0, '36 to 40': 38.0, '41 to 50': 45.5, '51 to 65': 58.0, 'over 65': 66.0}
NumberOfCars {'1 vehicle': 1.0, '2 vehicles': 2.0, '3 to 4': 3.5, '5 to 8': 6.5, 'more than 8': 9.0}


In [10]:
# convert
for col in range_cols:
    df[col] = df[col].apply(lambda v: parse_numeric_category(v, col)[0])

In [11]:
# check step 3
n_categories_after = df[range_cols].nunique()
merged = n_categories_before[n_categories_before != n_categories_after]
assert merged.empty, f"Categories collapsed into the same number: {merged.to_dict()}"
assert df[range_cols].notna().all().all(), "NaNs after conversion"
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in range_cols), "A range column isn't numeric"
print("✓ Step 3 | no merged categories, no NaNs, all numeric")

✓ Step 3 | no merged categories, no NaNs, all numeric


# step 4 --> split by year and group rare makes

In [12]:
train_df = df[df['Year'] == 1994].copy()
val_df   = df[df['Year'] == 1995].copy()
test_df  = df[df['Year'] == 1996].copy()   # DO NOT TOUCH until selection.md is written

# Rare makes → "Other". The threshold is counted on 1994 only, so later years don't leak in.
MIN_CLAIMS = 50
common_makes = train_df['Make'].value_counts()
common_makes = common_makes[common_makes >= MIN_CLAIMS].index
for d in (train_df, val_df, test_df):
    d['Make'] = d['Make'].where(d['Make'].isin(common_makes), 'Other')

X_train, y_train = train_df[features], train_df['FraudFound_P']
X_val,   y_val   = val_df[features],   val_df['FraudFound_P']

TEST_SCORED = False   # flipped to True only when 1996 is scored

In [13]:
# step 4 check
assert set(train_df['Year']) == {1994} and set(val_df['Year']) == {1995} and set(test_df['Year']) == {1996}, "A split has the wrong year"
assert len(train_df) + len(val_df) + len(test_df) == len(df), "Rows lost or duplicated in the split"
assert set(train_df.index).isdisjoint(val_df.index) and set(val_df.index).isdisjoint(test_df.index), "Splits overlap"
assert set(val_df['Make']) <= set(train_df['Make']), "1995 has a car make that 1994 never saw"
assert list(X_train.columns) == list(X_val.columns) == features, "Train/val columns don't match the feature list"
assert (train_df['Make'].value_counts().drop('Other', errors='ignore') >= MIN_CLAIMS).all(), "A rare make wasn't grouped"

print(pd.DataFrame({'claims': [len(train_df), len(val_df), len(test_df)],
                    'fraud rate': [d['FraudFound_P'].mean() for d in (train_df, val_df, test_df)]},
                   index=[1994, 1995, 1996]).round(4))
print("Makes kept:", sorted(train_df['Make'].unique()))
print("✓ Step 4")

      claims  fraud rate
1994    6141      0.0666
1995    5195      0.0579
1996    4083      0.0522
Makes kept: ['Accura', 'Chevrolet', 'Ford', 'Honda', 'Mazda', 'Other', 'Pontiac', 'Toyota', 'VW']
✓ Step 4


# step 5: build and train lr model

In [14]:
preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
])

logreg = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(class_weight='balanced', C=1.0,
                                 max_iter=3000, random_state=42)),
])

logreg.fit(X_train, y_train)                  # learns everything from 1994 only
lr_scores = logreg.predict_proba(X_val)[:, 1] # risk score for each 1995 claim

In [15]:
# step 5 check
age_imputer = logreg.named_steps['prep'].named_transformers_['num'].named_steps['impute']
assert np.isclose(age_imputer.statistics_[numeric_cols.index('Age')], X_train['Age'].median()), "Age median wasn't learned from 1994 only"

scaler = logreg.named_steps['prep'].named_transformers_['num'].named_steps['scale']
assert scaler.n_samples_seen_ == len(X_train), "Scaler saw more rows than the 1994 training set"

assert len(lr_scores) == len(y_val), "Wrong number of validation scores"
assert np.isfinite(lr_scores).all() and ((lr_scores >= 0) & (lr_scores <= 1)).all(), "Scores aren't valid probabilities"
assert lr_scores.std() > 0, "Model gives every claim the same score"
print(f"✓ Step 5 | {len(logreg[:-1].get_feature_names_out())} columns after one-hot encoding")

✓ Step 5 | 41 columns after one-hot encoding


# metric function evaluate

In [16]:
def evaluate(name, y, scores):
    y = np.asarray(y)                          # positions, not pandas labels
    scores = np.asarray(scores, dtype=float)   # accept lists, Series or arrays
    order = np.argsort(-scores, kind='stable') # riskiest claim first

    row = {'Approach': name, 'PR-AUC': average_precision_score(y, scores)}
    for frac in (0.10, 0.20):
        top = y[order[:int(np.ceil(frac * len(y)))]]   # labels of the reviewed claims
        row[f'Fraud caught @ top {int(frac*100)}%'] = top.sum() / y.sum()
        if frac == 0.20:
            p, r = top.mean(), top.sum() / y.sum()
            row['Precision @ top 20%'] = p
            row['F1 @ top 20%'] = 2*p*r / (p + r) if p + r else 0
    return row

In [17]:
# step 6 check
y_toy = np.array([1]*10 + [0]*90)          # 10% fraud
perfect = evaluate('perfect', y_toy, np.linspace(1, 0, 100))   # fraud ranked first
worst   = evaluate('worst',   y_toy, np.linspace(0, 1, 100))   # fraud ranked last

assert np.isclose(perfect['PR-AUC'], 1.0)
assert perfect['Fraud caught @ top 10%'] == 1.0 and perfect['Fraud caught @ top 20%'] == 1.0
assert np.isclose(perfect['Precision @ top 20%'], 0.5)    # 10 fraud out of 20 flagged
assert worst['Fraud caught @ top 10%'] == 0 and worst['Fraud caught @ top 20%'] == 0
print("✓ Step 6 | evaluate() gives the right answers on known cases")

✓ Step 6 | evaluate() gives the right answers on known cases


# comparison table step 7

In [18]:
def averaged(name, y, base_scores, n=50):
    """Average evaluate() over n random tie-breaks (or n random draws)."""
    rows = [evaluate(name, y, base_scores(np.random.default_rng(s))) for s in range(n)]
    return {'Approach': name, **pd.DataFrame(rows).drop(columns='Approach').mean().to_dict()}

# 1. Random picking: averaged over 50 seeds so one lucky draw can't mislead
random_row = averaged('Random picking', y_val, lambda rng: rng.random(len(y_val)))

# 2. Always say not fraud: flags nothing, so it catches nothing
never_row = {'Approach': 'Always say not fraud', 'PR-AUC': y_val.mean(),
             'Fraud caught @ top 10%': 0.0, 'Fraud caught @ top 20%': 0.0,
             'Precision @ top 20%': 0.0, 'F1 @ top 20%': 0.0}

# 3. Lookup rule: 1994 fraud rate per coverage x fault group, ties broken randomly
rates = train_df.groupby(['BasePolicy', 'Fault'])['FraudFound_P'].mean()
base_rate = y_train.mean()
lookup_scores = np.array([rates.get(k, base_rate)
                          for k in zip(val_df['BasePolicy'], val_df['Fault'])])
lookup_row = averaged('Lookup: coverage + fault', y_val,
                      lambda rng: lookup_scores + rng.random(len(y_val)) * 1e-9)

# 4. Your model
lr_row = evaluate('Logistic regression', y_val, lr_scores)

results = pd.DataFrame([random_row, never_row, lookup_row, lr_row])
display(results.round(3))

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%
0,Random picking,0.059,0.095,0.197,0.057,0.089
1,Always say not fraud,0.058,0.000,0.000,0.000,0.000
2,Lookup: coverage + fault,0.136,0.252,0.491,0.142,0.221
3,Logistic regression,0.143,0.292,0.535,0.155,0.240


In [19]:
# step 7 check
assert abs(random_row['Fraud caught @ top 10%'] - 0.10) < 0.02 and abs(random_row['Fraud caught @ top 20%'] - 0.20) < 0.02, "Random baseline is off"
assert lookup_scores.shape == (len(y_val),), "Lookup scores have the wrong shape"
assert len(rates) <= 6, "Lookup table has unexpected groups"
assert not np.isnan(lookup_scores).any(), "Some 1995 claims got no lookup score"
assert np.isclose(average_precision_score(y_val, np.zeros(len(y_val))), y_val.mean()), "Always-not-fraud PR-AUC should equal the fraud rate"
assert not TEST_SCORED, "1996 was scored before the model was chosen!"
print("✓ Step 7")

✓ Step 7


1995 (validation): “Reviewing the top 20% of claims catches 53.5% of fraud, compared with about 20% if we picked claims at random.” The official version of that sentence comes from 1996.

## look into gradient boosting, random forest, other models

step a1: comparison helper and random forest

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

SEEDS = [0, 1, 2, 3, 4]

def mean_row(name, rows):
    return {'Approach': name, **pd.DataFrame(rows).drop(columns='Approach').mean().to_dict()}

def compare(name, make_model):
    """Fit make_model(seed) for each seed on 1994; average metrics on 1995 (val) and 1994 (train)."""
    val_rows, train_rows = [], []
    for s in SEEDS:
        m = make_model(s)
        m.fit(X_train, y_train)
        val_rows.append(evaluate(name, y_val, m.predict_proba(X_val)[:, 1]))
        train_rows.append(evaluate(name, y_train, m.predict_proba(X_train)[:, 1]))
    row = mean_row(name, val_rows)
    row['Train: caught @ top 20%'] = mean_row(name, train_rows)['Fraud caught @ top 20%']
    return row

In [22]:
# tree processing and random forest
# Trees don't need scaling or one-hot: fill missing Age, give each category a number
tree_prep = lambda: ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols),
])

def make_rf(leaf):
    return lambda seed: Pipeline([
        ('prep', tree_prep()),
        ('model', RandomForestClassifier(n_estimators=500, min_samples_leaf=leaf,
                                         max_features='sqrt', class_weight='balanced_subsample',
                                         n_jobs=-1, random_state=seed)),
    ])

model_rows = [compare('Logistic regression', lambda s: clone(logreg))]
for leaf in [20, 50]:   # grid fixed in advance: two settings only
    model_rows.append(compare(f'Random forest (leaf={leaf})', make_rf(leaf)))

model_results = pd.DataFrame(model_rows)
display(model_results.round(3))

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%,Train: caught @ top 20%
0,Logistic regression,0.143,0.292,0.535,0.155,0.240,0.609
1,Random forest (leaf=20),0.144,0.293,0.527,0.153,0.237,0.788
2,Random forest (leaf=50),0.139,0.293,0.526,0.152,0.236,0.707


# gradient boosting

In [23]:
from sklearn.ensemble import HistGradientBoostingClassifier

cat_idx = list(range(len(numeric_cols), len(numeric_cols) + len(categorical_cols)))  # category positions after tree_prep

def make_hgb(depth):
    return lambda seed: Pipeline([
        ('prep', tree_prep()),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.05, max_depth=depth, min_samples_leaf=30,
            max_iter=500, early_stopping=True, validation_fraction=0.2,
            class_weight='balanced', categorical_features=cat_idx,
            random_state=seed)),
    ])

for depth in [2, 3]:   # grid fixed in advance
    model_rows.append(compare(f'Gradient boosting (depth={depth})', make_hgb(depth)))

model_results = pd.DataFrame(model_rows)
display(model_results.round(3))

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%,Train: caught @ top 20%
0,Logistic regression,0.143,0.292,0.535,0.155,0.240,0.609
1,Random forest (leaf=20),0.144,0.293,0.527,0.153,0.237,0.788
2,Random forest (leaf=50),0.139,0.293,0.526,0.152,0.236,0.707
3,Gradient boosting (depth=2),0.149,0.295,0.529,0.153,0.238,0.652
4,Gradient boosting (depth=3),0.158,0.301,0.538,0.156,0.242,0.677


## Explainable Boosting Machine

In [25]:
%pip install interpret

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

feature_types = ['continuous'] * len(numeric_cols) + ['nominal'] * len(categorical_cols)

def compare_ebm(name, interactions):
    val_rows, train_rows = [], []
    for s in SEEDS:
        m = ExplainableBoostingClassifier(feature_types=feature_types,
                                          interactions=interactions, random_state=s)
        m.fit(X_train, y_train, sample_weight=compute_sample_weight('balanced', y_train))
        val_rows.append(evaluate(name, y_val, m.predict_proba(X_val)[:, 1]))
        train_rows.append(evaluate(name, y_train, m.predict_proba(X_train)[:, 1]))
    row = mean_row(name, val_rows)
    row['Train: caught @ top 20%'] = mean_row(name, train_rows)['Fraud caught @ top 20%']
    return row

for k in [0, 10]:   # grid: no interactions vs up to 10 pairs
    model_rows.append(compare_ebm(f'EBM (interactions={k})', k))

model_results = pd.DataFrame(model_rows)
display(model_results.round(3))

c:\Users\amanm\AppData\Local\Programs\Python\Python312\Lib\site-packages\interpret\glassbox\_ebm\_ebm.py:871: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(
c:\Users\amanm\AppData\Local\Programs\Python\Python312\Lib\site-packages\interpret\glassbox\_ebm\_ebm.py:871: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(
c:\Users\amanm\AppData\Local\Programs\Python\Python312\Lib\site-packages\interpret\glassbox\_ebm\_ebm.py:8

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%,Train: caught @ top 20%
0,Logistic regression,0.143,0.292,0.535,0.155,0.240,0.609
1,Random forest (leaf=20),0.144,0.293,0.527,0.153,0.237,0.788
2,Random forest (leaf=50),0.139,0.293,0.526,0.152,0.236,0.707
3,Gradient boosting (depth=2),0.149,0.295,0.529,0.153,0.238,0.652
4,Gradient boosting (depth=3),0.158,0.301,0.538,0.156,0.242,0.677
5,EBM (interactions=0),0.146,0.306,0.522,0.151,0.234,0.631
6,EBM (interactions=10),0.155,0.310,0.546,0.158,0.245,0.692


# confusion matrix

In [27]:
from sklearn.metrics import confusion_matrix

def top_k_flags(scores, frac=0.20):
    scores = np.asarray(scores, dtype=float)
    n_flag = int(np.ceil(frac * len(scores)))
    flags = np.zeros(len(scores), dtype=int)
    flags[np.argsort(-scores, kind='stable')[:n_flag]] = 1   # top 20% → flagged
    return flags

flags = top_k_flags(lr_scores)
tn, fp, fn, tp = confusion_matrix(y_val, flags, labels=[0, 1]).ravel()

display(pd.DataFrame([[tn, fp], [fn, tp]],
                     index=['Actually honest', 'Actually fraud'],
                     columns=['Not flagged', 'Flagged (top 20%)']))

print(f"Claims reviewed:            {tp + fp:,} of {len(y_val):,}")
print(f"Fraud caught:               {tp} of {tp + fn} ({tp / (tp + fn):.1%})")
print(f"Fraud missed:               {fn}")
print(f"Honest claims flagged:      {fp} (false alarms)")
print(f"Honest claims cleared:      {tn}")
print(f"Flags that were real fraud: {tp / (tp + fp):.1%}  (about 1 in {round((tp + fp) / tp)})")

,Not flagged,Flagged (top 20%)
Actually honest,4016,878
Actually fraud,140,161


Claims reviewed:            1,039 of 5,195
Fraud caught:               161 of 301 (53.5%)
Fraud missed:               140
Honest claims flagged:      878 (false alarms)
Honest claims cleared:      4016
Flags that were real fraud: 15.5%  (about 1 in 6)


# without fault row -- missing feedback needed for 1995 table

In [28]:
cat_no_fault = [c for c in categorical_cols if c != 'Fault']
no_fault = Pipeline([
    ('prep', ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_no_fault)])),
    ('model', LogisticRegression(class_weight='balanced', C=1.0, max_iter=3000, random_state=42)),
])
no_fault.fit(train_df[numeric_cols + cat_no_fault], y_train)
nf_scores = no_fault.predict_proba(val_df[numeric_cols + cat_no_fault])[:, 1]

results = pd.concat([results, pd.DataFrame([evaluate('Logistic regression (no Fault)', y_val, nf_scores)])],
                    ignore_index=True)
display(results.round(3))

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%
0,Random picking,0.059,0.095,0.197,0.057,0.089
1,Always say not fraud,0.058,0.000,0.000,0.000,0.000
2,Lookup: coverage + fault,0.136,0.252,0.491,0.142,0.221
3,Logistic regression,0.143,0.292,0.535,0.155,0.240
4,Logistic regression (no Fault),0.104,0.233,0.399,0.115,0.179


Fault is definitely a need. Without fault, there was a loss of 13.6 points of fraud caught at 20%. 

quick test with no class weighting

In [29]:
lr_unweighted = clone(logreg).set_params(model__class_weight=None)
lr_unweighted.fit(X_train, y_train)
unw_scores = lr_unweighted.predict_proba(X_val)[:, 1]

display(pd.DataFrame([
    evaluate('LR, class_weight=balanced', y_val, lr_scores),
    evaluate('LR, no class weighting',    y_val, unw_scores),
]).round(3))

print(f"Average score, balanced:   {lr_scores.mean():.3f}")
print(f"Average score, unweighted: {unw_scores.mean():.3f}   (1995 fraud rate: {y_val.mean():.3f})")

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%
0,"LR, class_weight=balanced",0.143,0.292,0.535,0.155,0.240
1,"LR, no class weighting",0.143,0.282,0.518,0.150,0.233


Average score, balanced:   0.338
Average score, unweighted: 0.062   (1995 fraud rate: 0.058)
